# 🚦 Traffic Demand Prediction — XGBoost Model

In [ ]:
!pip install pygeohash

In [ ]:
!pip install pygeohash optuna -q
print(' Libraries installed')

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
N_TRIALS = 30

In [ ]:
import pandas as pd
import numpy as np
import pygeohash as geohash
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from xgboost import XGBRegressor

print('Imports done')

In [ ]:

TRAIN_PATH      = 'train.csv'
TEST_PATH       = 'test.csv'
SUBMISSION_PATH = 'sample_submission.csv'
OUTPUT_PATH     = 'submission.csv'


FEATURES = [
    'lat', 'lon',
    'day',
    'hour', 'minute', 'dayofweek',
    'month', 'is_weekend',
    'RoadType', 'NumberofLanes',
    'LargeVehicles', 'Landmarks',
    'Temperature', 'Weather'
]
TARGET = 'demand'


VAL_SIZE    = 0.2
RANDOM_SEED = 42


N_ESTIMATORS          = 1000
LEARNING_RATE         = 0.05
MAX_DEPTH             = 6
SUBSAMPLE             = 0.8
COLSAMPLE_BYTREE      = 0.8
EARLY_STOPPING_ROUNDS = 50
VERBOSE               = 100


print('Parameters set')

In [ ]:

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

print(f'Train shape : {train.shape}')
print(f'Test shape  : {test.shape}')

In [ ]:

def preprocess(df):
    df = df.copy()


    df['lat'] = df['geohash'].apply(lambda x: geohash.decode(x)[0])
    df['lon'] = df['geohash'].apply(lambda x: geohash.decode(x)[1])


    df['hour']       = df['timestamp'].apply(lambda x: int(str(x).split(':')[0]))
    df['minute']     = df['timestamp'].apply(lambda x: int(str(x).split(':')[1]))
    df['dayofweek']  = 0
    df['month']      = 0
    df['is_weekend'] = 0


    cat_cols = ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather']
    for col in cat_cols:
        if col in df.columns:
            df[col] = LabelEncoder().fit_transform(df[col].astype(str))


    df.drop(columns=['geohash', 'timestamp'], inplace=True)

    return df
train_clean = preprocess(train)
test_clean  = preprocess(test)

print(' Preprocessing done')
print(train_clean.head(3))

In [ ]:

X      = train_clean[FEATURES]
y      = train_clean[TARGET]
X_test = test_clean[FEATURES]

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y,
    test_size   = VAL_SIZE,
    random_state= RANDOM_SEED
)

print(f'Training size  : {X_tr.shape}')
print(f'Validation size: {X_val.shape}')
print(f'Test size      : {X_test.shape}')

In [ ]:
def objective(trial):
    params = {
        'n_estimators'         : trial.suggest_int('n_estimators', 200, 1000),
        'learning_rate'        : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth'            : trial.suggest_int('max_depth', 3, 10),
        'subsample'            : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree'     : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight'     : trial.suggest_int('min_child_weight', 1, 10),
        'gamma'                : trial.suggest_float('gamma', 0, 5),
        'reg_alpha'            : trial.suggest_float('reg_alpha', 0, 5),
        'reg_lambda'           : trial.suggest_float('reg_lambda', 0, 5),
        'random_state'         : RANDOM_SEED,
        'eval_metric'          : 'rmse',
        'early_stopping_rounds': 30
    }
    model = XGBRegressor(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    preds = model.predict(X_val)
    return r2_score(y_val, preds)

print(f' Starting tuning with {N_TRIALS} trials...')
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\n Best R²    : {study.best_value:.4f}')
print(f' Best Score : {max(0, 100 * study.best_value):.2f} / 100')
print('\n=== Best Parameters ===')
for key, val in study.best_params.items():
    print(f'  {key:25s} : {val}')

In [ ]:

best_params = study.best_params
best_params['random_state']          = RANDOM_SEED
best_params['eval_metric']           = 'rmse'
best_params['early_stopping_rounds'] = 50

model = XGBRegressor(**best_params)
model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=100)

val_preds = model.predict(X_val)
r2        = r2_score(y_val, val_preds)
score     = max(0, 100 * r2)

print(f'Final R²    : {r2:.4f}')
print(f'Final Score : {score:.2f} / 100')

In [ ]:

val_preds = model.predict(X_val)
r2        = r2_score(y_val, val_preds)
score     = max(0, 100 * r2)

print(f'Validation R²     : {r2:.4f}')
print(f'Competition Score : {score:.2f} / 100')

In [ ]:

importance_df = pd.DataFrame({
    'Feature'   : FEATURES,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

print('=== Feature Importance (higher = more useful) ===')
print(importance_df.to_string(index=False))

In [ ]:

test_preds = model.predict(X_test)

submission = pd.DataFrame({
    'Index' : test['Index'],
    'demand': test_preds
})

print(f'Submission shape: {submission.shape}')  # must be (41778, 2)
print(submission.head())

submission.to_csv(OUTPUT_PATH, index=False)
print(f'Saved to {OUTPUT_PATH}')